<a target="_blank" href="https://colab.research.google.com/github/LSSTC-DSFP/Session-25/tree/main/Day1/SupervisedMLProblems.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# From the lecture...

```python
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Construct features and labels
X = ...
y = ...

# Separate training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Instantiate a model
clf = DecisionTreeClassifier()

# Specify hyperparameter values to test
parameters = {'max_depth': range(1, 20),
              'criterion': ['gini', 'entropy'],
              'min_samples_leaf': range(1, 4)}

# Run grid search
gridsearch = GridSearchCV(clf, parameters, scoring='accuracy', cv=10)
gridsearch.fit(X_train, y_train)

# Evaluate performance
print(classification_report(y_test, gridsearch.best_estimator_.predict(X_test)))
```

# Problem 0

Load the Wisconsin Breast Cancer dataset into an X, y structure.

Establish training and testing data.

In [1]:
# This one I'll give the solution for since it will be the basis for the other
# problems we'll tackle.

from sklearn.datasets import load_breast_cancer

# let's first use a DataFrame to display the data so we can get a feel for it.
df = load_breast_cancer(as_frame=True)
df.frame.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [2]:
# The DataFrame was fun, but this is the solution to the problem
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Problem 1

Use the workflow from the lecture to train a `DecisionTreeClassifier` and evaluate its performance.

Print out the optimized hyperparameter settings you found.



# Problem 2

Choose another classifier and see if you can get higher accuracy. Use one of the following:

```python
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier  # <-- would recommend
```

For whichever one you choose, determine which hyperparameters you have available to you. To do this, take a quick look at either the scikit-learn docs or perform some introspection of the imported classes.

Heads up, the larger your grid search, the longer your runtime.

## Problem 2.5

The Random Forest performed the best. Why? It's just a bunch of DecisionTreeClassifiers under the hood, so what makes it out perform?

_Hint: theres a key word in the import statement of the RFC._

# Problem 3

Let's explore the `RandomForestClassifier` a little more.

Use your trained classifer from problem 2, or if you didn't choose the RFC, train one with `criterion='entropy', max_depth=15, n_estimators=20`.

The trained classifier will have an attribute called `feature_importances_`. **Determine what the 5 most important features in the dataset are.**

Classifier introspection is a very useful tool in science / astronomy applications. Reviewers, collaborators, and (hopefully) you as a scientist, will be interested in the key drivers your classifications are based upon.

## Problem 3.5

Make a pairplot of the data for the 5 most important features (or better yet, ask Gemini if working in Colab).

My prompt:
```
using matplotlib, write code for a corner plot to display the distributions of the [<REDACTED: list the features from problem 3 here>] columns of the dataset in this notebook. color the data points by their target_name.
```

Does it make sense that these features were the most important? Do you see separation between the benign and malignant populations?

# Problem 4

Overall goal: Plot a ROC Curve for your classifier. Add annotations to it that will make the plot publication ready.

### Step 1

Inspect the output of `clf.predict_proba(X)` (assuming your classifier is named `clf`).


There are two coloumns in the returned array. The first column is the probability (a statistician might get mad at me calling it a probability, but let's just treat it as such for now) of the example being the first class and the second column is the probablity of the example being the second class.

Now believe it or not, we aren't restricted to classifying examples as benign or malignant using a _probability_ threshold of 0.5. We can choose any threshold on the interval [0, 1] to make our classifications.

### Step 2

Get familiar with the Scikit-Learn `roc_curve` function.

Docs: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html#roc-curve

The `roc_curve` function from Scikit-Learn will return three things: `fpr`, `tpr`, and `thresholds`. You should interpret them as "for a given probability threshold, this is your false positive rate and true positive rate.

### Step 3

Use the `roc_curve` function to get the FPR, TPR, and thresholds arrays. Plot TPR as a function of FPR. Color by threshold if you're feeling fancy.

## Problem 4.5

The ROC curve is interpretted as "the closer I can get the elbow of my curve to the point (fpr=0, tpr=1), the better my classification is."

A standard way of measuring that is by computing the area under the curve. The closer to 1.0 the AUC is, the closer you are to perfect classifications.

**Calculate the ROC AUC, and add it to your previous graph in a legend.**

Hint: Of course, Scikit-Learn has a convenient way of doing this: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html#roc-auc-score

## Problem 4.5.5

Now what threshold should we choose? Let's think about the dataset. This is a toy problem, but one could imagine our classifications being used to make a breast cancer diagnosis for real human patients.

Would you rather prioritize a high TPR and tolerate an FPR? Would you rather prioritize a low FPR and compromise on TPR? Would you like to stay balanced and remain in the middle?

Personally, I would choose to err on the side where I classify a malignant tumor as benign as infrequently as possible, to avoid patients in need missing out on treatment. That is, at the minimum FPR for which TPR = 1.

**Choose an operating threshold, annotate your plot with a star (marker='*') at that threshold, and annotate the point with the FPR and TPR of that threshold.**